# MZM MLP PINN — Hyperparameter Tuning

## Overview
This notebook performs hyperparameter tuning for the Multi-Layer Perceptron Physics-Informed Neural Network (MLP-PINN) model for Mach-Zehnder Modulator (MZM) modeling.

### Hyperparameters Tuned
| Parameter | Description |
|---|---|
| `lambda_bw_mon` | Weight for BW monotonicity constraint ($\partial BW / \partial L \le 0$) |
| `lambda_IL_mon` | Weight for IL monotonicity constraint ($\partial IL / \partial L \ge 0$) |
| `lambda_vpiL` | Weight for $V_\pi \cdot L$ conservation constraint |
| `lambda_IL_offset_convex` | Weight for PN-offset convexity of IL ($\partial^2 IL / \partial PN_{offset} ^2 \ge 0$) |
| `lambda_Vpi_offset_convex` | Weight for PN-offset convexity of $V_\pi$ ($\partial^2 V_\pi / \partial PN_{offset} ^2 \ge 0$) |

We use **Optuna** for efficient Bayesian-style hyperparameter search with pruning of unpromising trials.

## Setup

In [ ]:
# # ==============================
# # Install Optuna
# # ==============================
# !pip install optuna

In [ ]:
# ==============================
# Import Required Libraries
# ==============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import re
import json
import itertools
from copy import deepcopy

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

import optuna
from optuna.trial import TrialState

# Reproducibility
random_state = 123
torch.manual_seed(random_state)
np.random.seed(random_state)

## Model Definitions

In [ ]:
class MLP5(nn.Module):
    def __init__(self, input_dim=8, output_dim=3, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 1350)
        self.bn1 = nn.BatchNorm1d(1350)

        self.fc2 = nn.Linear(1350, 200)
        self.bn2 = nn.BatchNorm1d(200)

        self.fc3 = nn.Linear(200, 300)
        self.bn3 = nn.BatchNorm1d(300)

        self.fc4 = nn.Linear(300, 350)
        self.bn4 = nn.BatchNorm1d(350)

        self.fc5 = nn.Linear(350, 300)
        self.bn5 = nn.BatchNorm1d(300)

        self.fc6 = nn.Linear(300, 200)
        self.bn6 = nn.BatchNorm1d(200)

        self.output = nn.Linear(200, output_dim)

        self.dropout = nn.Dropout(dropout)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):

        residual = x

        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)

        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)

        x = F.relu(self.bn3(self.fc3(x)))
        x = self.dropout(x)

        x = F.relu(self.bn4(self.fc4(x)))
        x = self.dropout(x)

        x = F.relu(self.bn5(self.fc5(x)))
        x = self.dropout(x)

        x = F.relu(self.bn6(self.fc6(x)))
        x = self.dropout(x)

        if residual.shape[1] < x.shape[1]:
            pad = x.shape[1] - residual.shape[1]
            residual = F.pad(residual, (0, pad))
        elif residual.shape[1] > x.shape[1]:
            residual = residual[:, :x.shape[1]]

        x = x + residual

        return self.output(x)

## Data Loading & Preprocessing

In [ ]:
# ==============================
# Load and Preprocess Data
# ==============================
# from google.colab import drive
# drive.mount('/content/drive')
# file_path = "/content/drive/MyDrive/MZM Data/Sim_generated_dataset.txt"
file_path = "Sim_generated_dataset.txt"

In [ ]:
with open(file_path) as f:
    cleaned = [re.sub(r'[\[\]]', '', line.strip()) for line in f]

df = pd.read_csv(StringIO("\n".join(cleaned)), header=None)
df.columns = [
    "PN_offset", "Bias_V", "Core_width", "P+_width", "N+_width",
    "P_width", "N_width", "Phase_length", "BW_3dB", "IL", "V_pi"
]

# Data cleaning: remove rows with extreme V_pi (>500, <0)
df_cleaned = df[(df['V_pi'] < 500) & (df['V_pi'] > 0)].copy()
print(f"Dataset size after cleaning: {len(df_cleaned)} samples")

# Split features / targets
feature_cols = df_cleaned.columns[:8]
target_cols = df_cleaned.columns[8:]

x = df_cleaned[feature_cols].values
y = df_cleaned[target_cols].values

# Train / test split
x_train_raw, x_test_raw, y_train_raw, y_test_raw = train_test_split(
    x, y, test_size=0.1, random_state=random_state
)

# ==============================
# Scaling (fit on TRAIN only)
# ==============================
scaler_x = StandardScaler()
scaler_y1 = StandardScaler()  # BW_3dB
scaler_y2 = StandardScaler()  # IL
scaler_y3 = StandardScaler()  # V_pi

x_train_scaled = scaler_x.fit_transform(x_train_raw)
y_train_scaled = np.hstack([
    scaler_y1.fit_transform(y_train_raw[:, 0:1]),
    scaler_y2.fit_transform(y_train_raw[:, 1:2]),
    scaler_y3.fit_transform(y_train_raw[:, 2:3]),
])

x_test_scaled = scaler_x.transform(x_test_raw)
y_test_scaled = np.hstack([
    scaler_y1.transform(y_test_raw[:, 0:1]),
    scaler_y2.transform(y_test_raw[:, 1:2]),
    scaler_y3.transform(y_test_raw[:, 2:3]),
])

# Convert to tensors
x_train_t = torch.from_numpy(x_train_scaled).float()
y_train_t = torch.from_numpy(y_train_scaled).float()
x_test_t  = torch.from_numpy(x_test_scaled).float()
y_test_t  = torch.from_numpy(y_test_scaled).float()

print(f"Train: {x_train_t.shape[0]} | Test: {x_test_t.shape[0]}")
df_cleaned.describe()

## Data Visualization

In [ ]:
# Combine all columns
all_cols = list(feature_cols) + list(target_cols)
n_cols = len(all_cols)

# Define grid size
n_grid_cols = 3
n_grid_rows = 4

fig, axes = plt.subplots(
    nrows=n_grid_rows,
    ncols=n_grid_cols,
    figsize=(6 * n_grid_cols, 4 * n_grid_rows)
)

axes = axes.flatten()

for i, col in enumerate(all_cols):
    axes[i].hist(df_cleaned[col], bins='auto', edgecolor='black')
    axes[i].set_title(f'{col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, linestyle='--', alpha=0.6)

# Remove empty subplots if any
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
x = df_cleaned['BW_3dB']
y = df_cleaned['V_pi']
c = df_cleaned['IL']

plt.figure(figsize=(10,6))
sc = plt.scatter(x, y, c=c, cmap='Blues', alpha=0.7, s=20, edgecolors='w', linewidth=0.3)

plt.xlabel('BW [GHz]')
plt.ylabel('V_pi [V]')
# plt.yscale('log')
plt.title('V_pi vs BW colored by IL')

# xlim and ylim matching the paper (page #5)
plt.xlim(15, 70)
plt.ylim(0, 50)

# Add colorbar
cbar = plt.colorbar(sc)
cbar.set_label('IL [dB]')

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = df_cleaned.columns[:8]
target_cols = df_cleaned.columns[8:]

n_features = len(feature_cols)
n_targets = len(target_cols)

fig, axes = plt.subplots(
    n_features,
    n_targets,
    figsize=(5 * n_targets, 5 * n_features)
)

for i, target_col in enumerate(target_cols):
    for j, feature_col in enumerate(feature_cols):
        ax = axes[j, i] if n_targets > 1 and n_features > 1 else (axes[j] if n_targets == 1 else axes[i])

        x_data = df_cleaned[feature_col].values
        y_data = df_cleaned[target_col].values

        # Perform polynomial curve fitting (degree=2)
        try:
            # Check for NaNs or Infs that could break polyfit
            valid_indices = ~np.isnan(x_data) & ~np.isinf(x_data) & ~np.isnan(y_data) & ~np.isinf(y_data)
            x_data_valid = x_data[valid_indices]
            y_data_valid = y_data[valid_indices]

            coeffs = np.polyfit(x_data_valid, y_data_valid, 2)
            poly = np.poly1d(coeffs)

            x_fit = np.linspace(x_data_valid.min(), x_data_valid.max(), 100)
            y_fit = poly(x_fit)

            # R² on valid points
            y_pred_valid = poly(x_data_valid)
            ss_res = np.sum((y_data_valid - y_pred_valid) ** 2)
            ss_tot = np.sum((y_data_valid - np.mean(y_data_valid)) ** 2)
            r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan

            a, b, c = coeffs
            poly_label = f"{a:.3e}x^2 + {b:.3e}x + {c:.3e}\nR²={r2:.4f}"

            ax.scatter(x_data, y_data, alpha=0.4, s=10)
            ax.plot(x_fit, y_fit, color='red', label=poly_label)
            ax.set_title(f'{target_col} vs {feature_col}', fontsize=10)
            ax.legend()

        except Exception as e:
            ax.scatter(x_data, y_data, alpha=0.4, s=10)
            ax.set_title(f'{target_col} vs {feature_col}\n(Fit Error: {e})', fontsize=10)

        ax.set_xlabel(feature_col, fontsize=8)
        ax.set_ylabel(target_col, fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle('Polynomial Curve Fits (Degree 2) for Output vs Input Parameters', y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Fit models for:
# 1) BW_3dB vs Phase_length with y = a*(1/x)
# 2) V_pi   vs Phase_length with y = a*(1/x)
# 3) IL     vs Phase_length with y = m*x + c

phase = df_cleaned["Phase_length"].to_numpy()
bw = df_cleaned["BW_3dB"].to_numpy()
vpi = df_cleaned["V_pi"].to_numpy()
il = df_cleaned["IL"].to_numpy()

mask = (
    np.isfinite(phase) & (phase > 0) &
    np.isfinite(bw) & np.isfinite(vpi) & np.isfinite(il)
)
phase_fit = phase[mask]
bw_fit = bw[mask]
vpi_fit = vpi[mask]
il_fit = il[mask]

# --- 1/x fits ---
u = 1.0 / phase_fit

# Least-squares fit through origin in transformed variable: y = a*u
a_bw = np.dot(u, bw_fit) / np.dot(u, u)
a_vpi = np.dot(u, vpi_fit) / np.dot(u, u)

bw_pred = a_bw / phase_fit
vpi_pred = a_vpi / phase_fit

# --- linear fit for IL ---
m_il, c_il = np.polyfit(phase_fit, il_fit, 1)
il_pred = m_il * phase_fit + c_il

# R^2
r2_bw = 1 - np.sum((bw_fit - bw_pred) ** 2) / np.sum((bw_fit - bw_fit.mean()) ** 2)
r2_vpi = 1 - np.sum((vpi_fit - vpi_pred) ** 2) / np.sum((vpi_fit - vpi_fit.mean()) ** 2)
r2_il = 1 - np.sum((il_fit - il_pred) ** 2) / np.sum((il_fit - il_fit.mean()) ** 2)

# Smooth curves for plotting
x_line = np.linspace(phase_fit.min(), phase_fit.max(), 300)
bw_line = a_bw / x_line
vpi_line = a_vpi / x_line
il_line = m_il * x_line + c_il

fig, axs = plt.subplots(1, 3, figsize=(21, 5))

axs[0].scatter(phase_fit, bw_fit, s=10, alpha=0.35, label="Data")
axs[0].plot(x_line, bw_line, "r", lw=2, label=f"Fit: y={a_bw:.3e}/x\nR²={r2_bw:.4f}")
axs[0].set_title("BW_3dB vs Phase_length (1/x fit)")
axs[0].set_xlabel("Phase_length")
axs[0].set_ylabel("BW_3dB")
axs[0].grid(True, alpha=0.3)
axs[0].legend()

axs[1].scatter(phase_fit, vpi_fit, s=10, alpha=0.35, label="Data")
axs[1].plot(x_line, vpi_line, "r", lw=2, label=f"Fit: y={a_vpi:.3e}/x\nR²={r2_vpi:.4f}")
axs[1].set_title("V_pi vs Phase_length (1/x fit)")
axs[1].set_xlabel("Phase_length")
axs[1].set_ylabel("V_pi")
axs[1].grid(True, alpha=0.3)
axs[1].legend()

axs[2].scatter(phase_fit, il_fit, s=10, alpha=0.35, label="Data")
axs[2].plot(x_line, il_line, "r", lw=2, label=f"Fit: y={m_il:.3e}x + {c_il:.3e}\nR²={r2_il:.4f}")
axs[2].set_title("IL vs Phase_length (linear fit)")
axs[2].set_xlabel("Phase_length")
axs[2].set_ylabel("IL")
axs[2].grid(True, alpha=0.3)
axs[2].legend()

plt.tight_layout()
plt.show()

print(f"BW_3dB fit: y = {a_bw:.6e}/x, R^2 = {r2_bw:.6f}")
print(f"V_pi fit:   y = {a_vpi:.6e}/x, R^2 = {r2_vpi:.6f}")
print(f"IL fit:     y = {m_il:.6e}x + {c_il:.6e}, R^2 = {r2_il:.6f}")

In [ ]:

# Fit 2nd degree polynomials for V_pi and IL vs PN_offset
pn_offset = df_cleaned["PN_offset"].to_numpy()
vpi = df_cleaned["V_pi"].to_numpy()
il = df_cleaned["IL"].to_numpy()

mask = (
    np.isfinite(pn_offset) & np.isfinite(vpi) & np.isfinite(il)
)
pn_offset_fit = pn_offset[mask]
vpi_fit = vpi[mask]
il_fit = il[mask]

# Fit 2nd degree polynomials
coeffs_vpi = np.polyfit(pn_offset_fit, vpi_fit, 2)
coeffs_il = np.polyfit(pn_offset_fit, il_fit, 2)

poly_vpi = np.poly1d(coeffs_vpi)
poly_il = np.poly1d(coeffs_il)

# Predictions
vpi_pred = poly_vpi(pn_offset_fit)
il_pred = poly_il(pn_offset_fit)

# R^2 scores
r2_vpi = 1 - np.sum((vpi_fit - vpi_pred) ** 2) / np.sum((vpi_fit - vpi_fit.mean()) ** 2)
r2_il = 1 - np.sum((il_fit - il_pred) ** 2) / np.sum((il_fit - il_fit.mean()) ** 2)

# Smooth curves for plotting
x_line = np.linspace(pn_offset_fit.min(), pn_offset_fit.max(), 300)
vpi_line = poly_vpi(x_line)
il_line = poly_il(x_line)

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

axs[0].scatter(pn_offset_fit, vpi_fit, s=10, alpha=0.35, label="Data")
axs[0].plot(x_line, vpi_line, "r", lw=2, label=f"Fit: {coeffs_vpi[0]:.3e}x² + {coeffs_vpi[1]:.3e}x + {coeffs_vpi[2]:.3e}\nR²={r2_vpi:.4f}")
axs[0].set_title("V_pi vs PN_offset (2nd degree fit)")
axs[0].set_xlabel("PN_offset")
axs[0].set_ylabel("V_pi")
axs[0].grid(True, alpha=0.3)
axs[0].legend()

axs[1].scatter(pn_offset_fit, il_fit, s=10, alpha=0.35, label="Data")
axs[1].plot(x_line, il_line, "r", lw=2, label=f"Fit: {coeffs_il[0]:.3e}x² + {coeffs_il[1]:.3e}x + {coeffs_il[2]:.3e}\nR²={r2_il:.4f}")
axs[1].set_title("IL vs PN_offset (2nd degree fit)")
axs[1].set_xlabel("PN_offset")
axs[1].set_ylabel("IL")
axs[1].grid(True, alpha=0.3)
axs[1].legend()

plt.tight_layout()
plt.show()

print(f"V_pi fit:   y = {coeffs_vpi[0]:.6e}x² + {coeffs_vpi[1]:.6e}x + {coeffs_vpi[2]:.6e}, R² = {r2_vpi:.6f}")
print(f"IL fit:     y = {coeffs_il[0]:.6e}x² + {coeffs_il[1]:.6e}x + {coeffs_il[2]:.6e}, R² = {r2_il:.6f}")

## GPU Setup

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

x_train_t = x_train_t.to(device)
y_train_t = y_train_t.to(device)
x_test_t  = x_test_t.to(device)
y_test_t  = y_test_t.to(device)

## Training Function
A self-contained training function that accepts all hyperparameters and returns the final train/test losses.

In [ ]:
def train_model(
    lambda_bw_mon,
    lambda_IL_mon,
    lambda_vpiL,
    lambda_IL_offset_convex,
    lambda_Vpi_offset_convex,
    # Fixed hyperparameters (change below if needed)
    D_i=8,
    D_o=3,
    dropout_rate=0.2,
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=0.05,
    betas=(0.9, 0.999),
    n_epoch=100,
    verbose=False,
    trial=None,  # Optuna trial for pruning
    early_stopping_patience=20,
    early_stopping_min_delta=1e-5,
    restore_best_weights=True,
    use_mixed_precision=True,
):
    """
    Build, train and evaluate a MLP-PINN model with the given hyperparameters.

    Returns
    -------
    dict with keys:
        'train_loss', 'test_loss',
        'train_history', 'test_history',
        'model_state'
    """
    # --- Build model ---
    model = MLP5(input_dim=D_i, output_dim=D_o, dropout=dropout_rate).to(device)

    n_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate,
        weight_decay=weight_decay, betas=betas,
    )
    loss_function = nn.MSELoss()

    # --- AMP setup ---
    amp_enabled = use_mixed_precision and (device.type == 'cuda')
    scaler = torch.GradScaler("cuda", enabled=amp_enabled)

    # --- DataLoader / TensorDataset ---
    train_dataset = TensorDataset(x_train_t, y_train_t)
    test_dataset = TensorDataset(x_test_t, y_test_t)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    train_hist = np.zeros(n_epoch)
    test_hist  = np.zeros(n_epoch)

    # Feature indices
    PN_OFFSET_IDX = 0
    PHASE_LENGTH_IDX = 7

    # --- Early stopping bookkeeping ---
    best_test_loss = float('inf')
    best_epoch = -1
    epochs_without_improvement = 0
    best_model_state = None
    stopped_epoch = n_epoch - 1

    # --- Training loop ---
    for epoch in range(n_epoch):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            x_batch.requires_grad_(True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type="cuda", enabled=amp_enabled):
                pred = model(x_batch)
                data_loss = loss_function(pred, y_batch)

                BW_pred  = pred[:, 0]
                IL_pred  = pred[:, 1]
                Vpi_pred = pred[:, 2]

                physics_loss = 0.0

                # 1) dBW/dL <= 0
                if lambda_bw_mon != 0:
                    grads = torch.autograd.grad(
                        BW_pred, x_batch,
                        grad_outputs=torch.ones_like(BW_pred),
                        create_graph=True,
                    )[0]
                    dBW_dL = grads[:, PHASE_LENGTH_IDX]
                    physics_loss += lambda_bw_mon * torch.mean(torch.relu(dBW_dL) ** 2)

                # 2) dIL/dL >= 0
                if lambda_IL_mon != 0:
                    grads = torch.autograd.grad(
                        IL_pred, x_batch,
                        grad_outputs=torch.ones_like(IL_pred),
                        create_graph=True,
                    )[0]
                    dIL_dL = grads[:, PHASE_LENGTH_IDX]
                    physics_loss += lambda_IL_mon * torch.mean(torch.relu(-dIL_dL) ** 2)

                # 3) d(Vpi*L)/dL <= 0
                if lambda_vpiL != 0:
                    grads = torch.autograd.grad(
                        Vpi_pred, x_batch,
                        grad_outputs=torch.ones_like(Vpi_pred),
                        create_graph=True,
                    )[0]
                    dVpiL_dL = grads[:, PHASE_LENGTH_IDX]
                    physics_loss += lambda_vpiL * torch.mean(torch.relu(dVpiL_dL) ** 2)

                # 4) d²IL/d(PN_offset)² >= 0  (convex IL vs PN_offset)
                if lambda_IL_offset_convex != 0:
                    grads1 = torch.autograd.grad(
                        IL_pred, x_batch,
                        grad_outputs=torch.ones_like(IL_pred),
                        create_graph=True,
                    )[0]
                    dIL_dPN = grads1[:, PN_OFFSET_IDX]
                    grads2 = torch.autograd.grad(
                        dIL_dPN, x_batch,
                        grad_outputs=torch.ones_like(dIL_dPN),
                        create_graph=True,
                    )[0]
                    d2IL_dPN2 = grads2[:, PN_OFFSET_IDX]
                    physics_loss += lambda_IL_offset_convex * torch.mean(torch.relu(-d2IL_dPN2) ** 2)

                # 5) d²Vpi/d(PN_offset)² >= 0  (convex Vpi vs PN_offset)
                if lambda_Vpi_offset_convex != 0:
                    grads1 = torch.autograd.grad(
                        Vpi_pred, x_batch,
                        grad_outputs=torch.ones_like(Vpi_pred),
                        create_graph=True,
                    )[0]
                    dVpi_dPN = grads1[:, PN_OFFSET_IDX]
                    grads2 = torch.autograd.grad(
                        dVpi_dPN, x_batch,
                        grad_outputs=torch.ones_like(dVpi_dPN),
                        create_graph=True,
                    )[0]
                    d2Vpi_dPN2 = grads2[:, PN_OFFSET_IDX]
                    physics_loss += lambda_Vpi_offset_convex * torch.mean(torch.relu(-d2Vpi_dPN2) ** 2)

                loss = data_loss + physics_loss

            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

        # --- Evaluation with DataLoaders ---
        model.eval()
        train_loss_sum, train_count = 0.0, 0
        test_loss_sum, test_count = 0.0, 0

        with torch.no_grad():
            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                with torch.autocast(device_type="cuda", enabled=amp_enabled):
                    l = loss_function(model(xb), yb)
                bs = xb.size(0)
                train_loss_sum += l.item() * bs
                train_count += bs

            for xb, yb in test_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                with torch.autocast(device_type="cuda", enabled=amp_enabled):
                    l = loss_function(model(xb), yb)
                bs = xb.size(0)
                test_loss_sum += l.item() * bs
                test_count += bs

        train_loss = train_loss_sum / max(train_count, 1)
        test_loss = test_loss_sum / max(test_count, 1)
        train_hist[epoch] = train_loss
        test_hist[epoch] = test_loss

        if verbose and (epoch % 5 == 0):
            print(f"  Epoch {epoch:4d} | Train {train_loss:.6f} | Test {test_loss:.6f}")

        # Optuna pruning
        if trial is not None:
            trial.report(test_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        # Early stopping (based on test loss)
        if test_loss < (best_test_loss - early_stopping_min_delta):
            best_test_loss = test_loss
            best_epoch = epoch
            epochs_without_improvement = 0
            if restore_best_weights:
                best_model_state = deepcopy(model.state_dict())
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stopping_patience:
            stopped_epoch = epoch
            if verbose:
                print(
                    f"Early stopping at epoch {epoch} (best epoch: {best_epoch}, best test: {best_test_loss:.6f})"
                )
            break

    # Keep only executed part of histories
    executed_epochs = stopped_epoch + 1
    train_hist = train_hist[:executed_epochs]
    test_hist = test_hist[:executed_epochs]

    # Restore best weights if requested and available
    if restore_best_weights and best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Re-evaluate final model (restored best if enabled)
    model.eval()
    train_loss_sum, train_count = 0.0, 0
    test_loss_sum, test_count = 0.0, 0
    with torch.no_grad():
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            with torch.autocast(device_type="cuda", enabled=amp_enabled):
                l = loss_function(model(xb), yb)
            bs = xb.size(0)
            train_loss_sum += l.item() * bs
            train_count += bs

        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            with torch.autocast(device_type="cuda", enabled=amp_enabled):
                l = loss_function(model(xb), yb)
            bs = xb.size(0)
            test_loss_sum += l.item() * bs
            test_count += bs

    final_train_loss = train_loss_sum / max(train_count, 1)
    final_test_loss = test_loss_sum / max(test_count, 1)

    return {
        'train_loss': final_train_loss,
        'test_loss':  final_test_loss,
        'best_test_loss': best_test_loss,
        'best_epoch': best_epoch,
        'stopped_epoch': stopped_epoch,
        'train_history': train_hist,
        'test_history':  test_hist,
        'model_state': deepcopy(model.state_dict()),
        'n_params': n_params,
    }

## Hyperparameter Search Configuration

Define the search space for all tuned hyperparameters.

In [ ]:
# ==============================
# Search Space Definition
# ==============================

# Number of Optuna trials
N_OPTUNA_TRIALS = 100

# Training epochs per trial (lower for faster search; retrain best with more)
SEARCH_EPOCHS = 40

# Final retraining epochs for the best config
FINAL_EPOCHS = 400

## Optuna Search
Uses TPE sampler and median pruner.

In [ ]:
def objective(trial):
    # --- Sample hyperparameters ---
    lambda_bw_mon = trial.suggest_float("lambda_bw_mon", 0.0, 1.0)
    lambda_IL_mon = trial.suggest_float("lambda_IL_mon", 0.0, 1.0)
    lambda_vpiL   = trial.suggest_float("lambda_vpiL", 0.0, 1.0)
    lambda_IL_offset_convex = trial.suggest_float("lambda_IL_offset_convex", 0.0, 1.0)
    lambda_Vpi_offset_convex = trial.suggest_float("lambda_Vpi_offset_convex", 0.0, 1.0)

    result = train_model(
        lambda_bw_mon=lambda_bw_mon,
        lambda_IL_mon=lambda_IL_mon,
        lambda_vpiL=lambda_vpiL,
        lambda_IL_offset_convex=lambda_IL_offset_convex,
        lambda_Vpi_offset_convex=lambda_Vpi_offset_convex,
        n_epoch=SEARCH_EPOCHS,
        verbose=False,
        trial=trial,
    )
    return result['test_loss']

In [ ]:
study = optuna.create_study(
    study_name="MLP_PINN_Hyperparam_Search",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=random_state),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=20),
)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

# --- Best trial summary ---
best = study.best_trial
print("\n" + "="*50)
print(f"Best test loss: {best.value:.6f}")
print("Best hyperparameters:")
for k, v in best.params.items():
    print(f"  {k} = {v}")
print("="*50)

## Results Analysis & Visualization

In [ ]:
# ==============================
# Collect best hyperparameters
# ==============================
try:
    bp = study.best_trial.params
    best_config = {
        'lambda_bw_mon': bp['lambda_bw_mon'],
        'lambda_IL_mon': bp['lambda_IL_mon'],
        'lambda_vpiL':   bp['lambda_vpiL'],
        'lambda_IL_offset_convex': bp['lambda_IL_offset_convex'],
        'lambda_Vpi_offset_convex': bp['lambda_Vpi_offset_convex'],
    }
except:
    best_config = {
        'lambda_bw_mon': 0.8377144489359248,
        'lambda_IL_mon': 0.25895168449143524,
        'lambda_vpiL': 0.7592177021321725,
        'lambda_IL_offset_convex': 0.1,
        'lambda_Vpi_offset_convex': 0.1,
    }

In [ ]:
# ==============================
# Optuna Visualizations
# ==============================
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
)

fig = plot_optimization_history(study)
plt.title("Optimization History")
plt.tight_layout()
plt.show()

fig = plot_param_importances(study)
plt.tight_layout()
plt.show()

fig = plot_parallel_coordinate(study)
plt.title("Parallel Coordinate Plot")
plt.tight_layout()
plt.show()

fig = plot_slice(study)
# plt.tight_layout()
plt.show()

## Bootstrapping Best Model (Full Epochs)
Train a bunch of models with the best configuration and more epochs to get the estimator distribution, then compare it to that of the MLP without the PINN.

In [ ]:
N_BOOTSTRAP_MODELS = 50

In [ ]:
bootstrapped_results_pinn = []
bootstrapped_model_states_pinn = []

# Store original training data globally to restore after each bootstrap iteration
# This is a workaround as train_model currently uses global x_train_t, y_train_t
original_x_train_t = x_train_t
original_y_train_t = y_train_t

print(f"Training {N_BOOTSTRAP_MODELS} models using bootstrapping with {FINAL_EPOCHS} epochs each...")

for i in range(N_BOOTSTRAP_MODELS):
    print(f"\n--- Bootstrapping Model {i+1}/{N_BOOTSTRAP_MODELS} ---")

    # Generate bootstrap indices (sample with replacement)
    bootstrap_indices = torch.randint(
        0, len(original_x_train_t), (int(0.63 * len(original_x_train_t)),),
        device=device
    )

    # Create bootstrapped training datasets
    x_train_bootstrap = original_x_train_t[bootstrap_indices]
    y_train_bootstrap = original_y_train_t[bootstrap_indices]

    # Temporarily assign bootstrapped data to global variables for train_model function
    # WARNING: This is a hack to avoid modifying the train_model signature.
    x_train_t = x_train_bootstrap
    y_train_t = y_train_bootstrap

    try:
        bootstrap_result = train_model(
            lambda_bw_mon=best_config['lambda_bw_mon'],
            lambda_IL_mon=best_config['lambda_IL_mon'],
            lambda_vpiL=best_config['lambda_vpiL'],
            lambda_IL_offset_convex=best_config['lambda_IL_offset_convex'],
            lambda_Vpi_offset_convex=best_config['lambda_Vpi_offset_convex'],
            n_epoch=FINAL_EPOCHS,
            verbose=True,
        )

        bootstrapped_results_pinn.append(bootstrap_result)
        bootstrapped_model_states_pinn.append(bootstrap_result['model_state'])

        print(f"  Model {i+1} Final Train Loss: {bootstrap_result['train_loss']:.6f}")
        print(f"  Model {i+1} Final Test  Loss: {bootstrap_result['test_loss']:.6f}")

    except RuntimeError as e:
        print(f"  Model {i+1} training failed: {e}")
    finally:
        # Restore original training data globally
        x_train_t = original_x_train_t
        y_train_t = original_y_train_t

print("\n--- Bootstrapping Summary ---")
if bootstrapped_results_pinn:
    avg_train_loss_pinn = np.mean([r['train_loss'] for r in bootstrapped_results_pinn])
    avg_test_loss_pinn = np.mean([r['test_loss'] for r in bootstrapped_results_pinn])
    print(f"Average Train Loss across {N_BOOTSTRAP_MODELS} models: {avg_train_loss_pinn:.6f}")
    print(f"Average Test  Loss across {N_BOOTSTRAP_MODELS} models: {avg_test_loss_pinn:.6f}")
else:
    print("No models were successfully trained during bootstrapping.")

# You now have `bootstrapped_results` and `bootstrapped_model_states`
# for further analysis, e.g., ensemble predictions or variability studies.


In [ ]:
bootstrapped_results_mlp = []
bootstrapped_model_states_mlp = []

# Store original training data globally to restore after each bootstrap iteration
# This is a workaround as train_model currently uses global x_train_t, y_train_t
original_x_train_t = x_train_t
original_y_train_t = y_train_t

print(f"Training {N_BOOTSTRAP_MODELS} models using bootstrapping with {FINAL_EPOCHS} epochs each...")

for i in range(N_BOOTSTRAP_MODELS):
    print(f"\n--- Bootstrapping Model {i+1}/{N_BOOTSTRAP_MODELS} ---")

    # Generate bootstrap indices (sample with replacement)
    bootstrap_indices = torch.randint(
        0, len(original_x_train_t), (int(0.63 * len(original_x_train_t)),),
        device=device
    )

    # Create bootstrapped training datasets
    x_train_bootstrap = original_x_train_t[bootstrap_indices]
    y_train_bootstrap = original_y_train_t[bootstrap_indices]

    # Temporarily assign bootstrapped data to global variables for train_model function
    # WARNING: This is a hack to avoid modifying the train_model signature.
    x_train_t = x_train_bootstrap
    y_train_t = y_train_bootstrap

    try:
        bootstrap_result = train_model(
            lambda_bw_mon=0,
            lambda_IL_mon=0,
            lambda_vpiL=0,
            lambda_IL_offset_convex=0,
            lambda_Vpi_offset_convex=0,
            n_epoch=FINAL_EPOCHS,
            verbose=True,
        )

        bootstrapped_results_mlp.append(bootstrap_result)
        bootstrapped_model_states_mlp.append(bootstrap_result['model_state'])

        print(f"  Model {i+1} Final Train Loss: {bootstrap_result['train_loss']:.6f}")
        print(f"  Model {i+1} Final Test  Loss: {bootstrap_result['test_loss']:.6f}")

    except RuntimeError as e:
        print(f"  Model {i+1} training failed: {e}")
    finally:
        # Restore original training data globally
        x_train_t = original_x_train_t
        y_train_t = original_y_train_t

print("\n--- Bootstrapping Summary ---")
if bootstrapped_results_mlp:
    avg_train_loss_mlp = np.mean([r['train_loss'] for r in bootstrapped_results_mlp])
    avg_test_loss_mlp = np.mean([r['test_loss'] for r in bootstrapped_results_mlp])
    print(f"Average Train Loss across {N_BOOTSTRAP_MODELS} models: {avg_train_loss_mlp:.6f}")
    print(f"Average Test  Loss across {N_BOOTSTRAP_MODELS} models: {avg_test_loss_mlp:.6f}")
else:
    print("No models were successfully trained during bootstrapping.")

# You now have `bootstrapped_results` and `bootstrapped_model_states`
# for further analysis, e.g., ensemble predictions or variability studies.



In [ ]:
# print(f"Retraining best config for {FINAL_EPOCHS} epochs ...\n")

# final_result = train_model(
#     lambda_bw_mon=best_config['lambda_bw_mon'],
#     lambda_IL_mon=best_config['lambda_IL_mon'],
#     lambda_vpiL=best_config['lambda_vpiL'],
#     lambda_smooth=best_config['lambda_smooth'],
#     n_epoch=FINAL_EPOCHS,
#     verbose=True,
# )

# print(f"\nFinal Train Loss: {final_result['train_loss']:.6f}")
# print(f"Final Test  Loss: {final_result['test_loss']:.6f}")
# print(f"Model parameters: {final_result['n_params']}")

In [ ]:
# # ==============================
# # Training Curves for Best Model
# # ==============================
# fig, ax = plt.subplots(figsize=(8, 5))
# ax.plot(final_result['train_history'], 'r-', label='Train')
# ax.plot(final_result['test_history'],  'b-', label='Test')
# ax.set_xlabel('Epoch')
# ax.set_ylabel('MSE')
# ax.set_title(
#     f"Best Model — Train {final_result['train_loss']:.5f}, "
#     f"Test {final_result['test_loss']:.5f}"
# )
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

In [ ]:
N_ROWS = (N_BOOTSTRAP_MODELS + 1) // 2
N_COLS = 5

fig, axes = plt.subplots(
    N_ROWS,
    N_COLS,
    figsize=(12, 4 * N_ROWS),
    squeeze=False
)
axes = axes.flatten()

for i, result in enumerate(bootstrapped_results_mlp):
    ax = axes[i]
    ax.plot(result['train_history'], 'r-', label='Train')
    ax.plot(result['test_history'],  'b-', label='Test')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.set_title(
        f"Bootstrap Model {i+1} — Train {result['train_loss']:.5f}, "
        f"Test {result['test_loss']:.5f}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide any unused subplots
for j in range(len(bootstrapped_results_mlp), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.suptitle('Training Curves for Bootstrapped Models', y=1.02, fontsize=16)
plt.show()

## Per-Output Error Analysis

In [ ]:
# # ==============================
# # Load best model and evaluate per-output
# # ==============================
# best_model = MLP5().to(device)
# best_model.load_state_dict(final_result['model_state'])
# best_model.eval()

# with torch.no_grad():
#     pred_test = best_model(x_test_t).cpu().numpy()

# y_test_np = y_test_t.cpu().numpy()
# target_names = ['BW_3dB', 'IL', 'V_pi']

# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# for i, (name, ax) in enumerate(zip(target_names, axes)):
#     ax.scatter(y_test_np[:, i], pred_test[:, i], alpha=0.4, s=10)
#     lims = [
#         min(y_test_np[:, i].min(), pred_test[:, i].min()),
#         max(y_test_np[:, i].max(), pred_test[:, i].max()),
#     ]
#     ax.plot(lims, lims, 'r--', lw=1)
#     mse_i = np.mean((y_test_np[:, i] - pred_test[:, i])**2)
#     ax.set_title(f"{name} — MSE: {mse_i:.5f}")
#     ax.set_xlabel('Simulated (scaled)')
#     ax.set_ylabel('Predicted (scaled)')
#     ax.grid(True, alpha=0.3)

# plt.suptitle('Prediction vs Simulation', fontsize=14, y=1.02)
# plt.tight_layout()
# plt.show()

In [ ]:
# Extract test MSEs from bootstrapped results
test_mses = [result['test_loss'] for result in bootstrapped_results_pinn]

# Create histogram
plt.figure(figsize=(8, 6))
plt.hist(test_mses, bins='auto', edgecolor='black', alpha=0.7, density=True)
sns.kdeplot(test_mses, color='blue')
plt.title('Distribution of Test MSE across 50 PINN Bootstrapped Models')
plt.xlabel('Test Mean Squared Error (MSE)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

In [ ]:
# Extract test MSEs from bootstrapped results
test_mses = [result['test_loss'] for result in bootstrapped_results_mlp]

# Create histogram
plt.figure(figsize=(8, 6))
plt.hist(test_mses, bins='auto', color='red', edgecolor='black', alpha=0.7, density=True)
sns.kdeplot(test_mses, color='red')
plt.title('Distribution of Test MSE across 50 MLP Bootstrapped Models')
plt.xlabel('Test Mean Squared Error (MSE)')
plt.ylabel('Density')
plt.tight_layout()
plt.show()

## Save Best Configuration

In [ ]:
# # ==============================
# # Save results
# # ==============================
# save_dict = {
#     'best_config': {k: (v if not isinstance(v, list) else v)
#                     for k, v in best_config.items()},
#     'final_train_loss': float(final_result['train_loss']),
#     'final_test_loss':  float(final_result['test_loss']),
#     'n_params': final_result['n_params'],
# }

# with open('best_hyperparams.json', 'w') as f:
#     json.dump(save_dict, f, indent=2)
# print("Saved to best_hyperparams.json")

# # Save model weights
# torch.save(final_result['model_state'], 'best_model.pt')
# print("Saved model weights to best_model.pt")

# print("\nBest configuration summary:")
# print(json.dumps(save_dict, indent=2))